In [3]:
import os
import subprocess
import sys

# 可选：在系统环境变量中设置 DOCTAMPER_ROOT 指向仓库根目录；否则从当前工作目录向上查找 models/eval_dtd.py
def _find_doctamper_root():
    p = os.path.abspath(os.getcwd())
    for _ in range(8):
        if os.path.isfile(os.path.join(p, "models", "eval_dtd.py")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    raise FileNotFoundError(
        "未找到 DocTamper 仓库（需要存在 models/eval_dtd.py）。"
        "请在仓库根目录启动 Jupyter，或先 os.chdir 到该目录，或设置环境变量 DOCTAMPER_ROOT。"
    )


PROJECT_ROOT = os.path.abspath(os.environ.get("DOCTAMPER_ROOT") or _find_doctamper_root())
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.chdir(MODELS_DIR)
sys.path.insert(0, MODELS_DIR)

# 若 LMDB 在仓库根目录而非 models 下，尝试创建目录联接（Windows: mklink /J；失败则请手动复制/联接到 models/DocTamperV1-FCD）
_fcd_at_root = os.path.join(PROJECT_ROOT, "DocTamperV1-FCD", "data.mdb")
_fcd_in_models = os.path.join(MODELS_DIR, "DocTamperV1-FCD")
if os.path.isfile(_fcd_at_root) and not os.path.exists(_fcd_in_models):
    try:
        if os.name == "nt":
            subprocess.run(
                [
                    "cmd",
                    "/c",
                    "mklink",
                    "/J",
                    _fcd_in_models,
                    os.path.join(PROJECT_ROOT, "DocTamperV1-FCD"),
                ],
                check=True,
            )
        else:
            os.symlink(
                os.path.join(PROJECT_ROOT, "DocTamperV1-FCD"),
                _fcd_in_models,
                target_is_directory=True,
            )
    except Exception as _e:
        print("未能在 models 下创建 DocTamperV1-FCD 联接（可忽略若你已放在 models 内）:", _e)

# 与 eval_dtd.py 一致：在 models 目录下需要已有下列文件（请自行从本机准备，不执行 clone/pip）
_checks = [
    ("LMDB", os.path.join(MODELS_DIR, "DocTamperV1-FCD", "data.mdb")),
    ("qt_table.pk", os.path.join(MODELS_DIR, "qt_table.pk")),
    ("DTD 权重", os.path.join(MODELS_DIR, "pths", "dtd_doctamper.pth")),
    ("vph_imagenet.pt", os.path.join(MODELS_DIR,"pths", "vph_imagenet.pt")),
    ("swin_imagenet.pt", os.path.join(MODELS_DIR, "pths", "swin_imagenet.pt")),
    ("pks/FCD", os.path.join(MODELS_DIR, "pks", "DocTamperV1-FCD_75.pk")),
]
for name, path in _checks:
    assert os.path.exists(path), f"缺少 {name}: {path}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("工作目录（已 chdir）:", MODELS_DIR)
print("本地资源检查通过（未执行 pip / git clone / Drive 挂载）。")

PROJECT_ROOT: /home/nanxin/workspace/doctamper/DocTamper
工作目录（已 chdir）: /home/nanxin/workspace/doctamper/DocTamper/models
本地资源检查通过（未执行 pip / git clone / Drive 挂载）。


### 本地模式说明

- **依赖**：PyTorch、CUDA（可选）、`jpegio`、`lmdb` 及 `models/requirements.txt` 中的包需已在当前 Python 环境中安装；本 notebook **不包含** `pip install` / `git clone` / Google Drive。
- **数据与权重**：`pths/dtd_doctamper.pth`、`vph_imagenet.pt`、`swin_imagenet.pt`、`qt_table.pk`、`pks/` 等须在 `models` 下。`DocTamperV1-FCD` 也须在 `models/DocTamperV1-FCD`（若你放在仓库根目录，初始化格会尝试用目录联接/junction 指过去；失败时请手动复制或联接）。
- **DocTamperV1-SCD**：将 LMDB 目录放在 `models/DocTamperV1-SCD/`，并准备 `pks/DocTamperV1-SCD_75.pk` 后再运行 SCD 评测格。

In [5]:
import subprocess

assert os.path.isfile(os.path.join(MODELS_DIR, "eval_dtd.py")), "请先运行初始化单元格"
_env = os.environ.copy()
_env["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(
    [
        sys.executable,
        "eval_dtd.py",
        "--lmdb_name",
        "DocTamperV1-FCD",
        "--pth",
        "pths/dtd_doctamper.pth",
        "--minq",
        "75",
    ],
    cwd=MODELS_DIR,
    env=_env,
    check=True,
)

/home/nanxin/workspace/doctamper/DocTamper/models/dtd.py:423: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
/home/nanxin/miniconda3/envs/torch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0%|          | 0/334 [00:00<?, ?it/s]/home/nanxin/miniconda3/envs/torch_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller tha

KeyboardInterrupt: 

In [ ]:
import subprocess

_scd = os.path.join(MODELS_DIR, "DocTamperV1-SCD", "data.mdb")
_scd_pk = os.path.join(MODELS_DIR, "pks", "DocTamperV1-SCD_75.pk")
assert os.path.isfile(_scd), f"请将 DocTamperV1-SCD（LMDB，含 data.mdb）放到: {os.path.dirname(_scd)}"
assert os.path.isfile(_scd_pk), f"缺少 SCD 预计算 pks 文件: {_scd_pk}"

_env = os.environ.copy()
_env["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(
    [
        sys.executable,
        "eval_dtd.py",
        "--lmdb_name",
        "DocTamperV1-SCD",
        "--pth",
        "pths/dtd_doctamper.pth",
        "--minq",
        "75",
    ],
    cwd=MODELS_DIR,
    env=_env,
    check=True,
)